# M3 variants: evaluation inside the final pipeline and selection of the final M3

**Run from the repository root (`Final_Model/`)** with the same Python environment as `Train_3SubModels.ipynb`
(`min16_lib.py`, `config.json`, `Train/` and `Test/` must be next to this notebook).

**Question.** The third sub-model (M3) is a collaborative-filtering (similar-participant) model. This notebook evaluates the
12 M3 variants of Table 5 of the article **inside the final pipeline** (same harmonisation, folds, M1, M2 and meta-learner),
bootstraps the differences in ensemble AUC, ranks the variants, and checks the final ensemble against the Brier rule
(the ensemble's Brier score must not exceed the best sub-model's).

**Protocol (identical to the primary analysis of `Train_3SubModels.ipynb`).** Original cohort (PATNO < 10000), participant-level
grouped `StratifiedGroupKFold` 5 x 3, participant- and visit-level harmonisation and class-balanced imputation inside every training fold,
cross-fitted L2 logistic-regression meta-learner on clipped logits, Youden threshold from training out-of-fold predictions.
Selection uses **training out-of-fold** results only; held-out test numbers are reported for confirmation.

| Part | What it does | Runtime (approx.) |
|---|---|---|
| A | Cache everything that does not depend on M3 (folds, harmonised training data, M1 and M2 out-of-fold/test predictions) | 15-30 min |
| B | Collaborative-filtering library with the design options and the 12 variants | seconds |
| C | Fit the 12 variants in every fold and evaluate them inside the ensemble | 10-30 min |
| D | Paired bootstrap and ranking (`top12_m3_variants.csv`) | 2 min |
| E | Final model (variant V4) and Brier-rule check from the saved results of `Train_3SubModels.ipynb` | seconds |

Set `QUICK = True` below for a smoke test with three variants. Outputs go to `M3_Redesign_Results/`; the cache in
`M3_Redesign_Results/cache/` contains participant-level data and must **not** be published.


In [ ]:
import os, sys, json, pickle, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

BASE = os.getcwd()                       # run from Final_Model/
sys.path.insert(0, BASE)
import min16_lib as lib

QUICK = False                            # True = smoke test with a few variants
N_JOBS = 2                               # parallel workers for the fold loop
OUT = os.path.join(BASE, "M3_Redesign_Results"); CACHE = os.path.join(OUT, "cache")
os.makedirs(CACHE, exist_ok=True)

CFG = json.load(open(os.path.join(BASE, "config.json")))
FEATURES = CFG["features"]; COLS = FEATURES + ["age", "sex"]
def _p(path): return path if os.path.isabs(path) else os.path.join(BASE, path)
TRAIN_FILE, TEST_FILE = _p(CFG["train_file"]), _p(CFG["test_file"])
print("train:", TRAIN_FILE, "\ntest :", TEST_FILE, "\nfeatures:", len(FEATURES))

## Part A. Cache the parts of the pipeline that do not depend on M3

M1 (per-feature LR + SVM + XGBoost trio with an XGBoost stacker) and M2 (global XGBoost) are fitted once per fold on the
harmonised training rows and their out-of-fold probabilities are cached. M3 candidates are then swapped in without refitting M1 and M2.
This mirrors `min16_lib.run_oof` and `train_final` (primary analysis).

In [ ]:
STAGE_A = os.path.join(CACHE, "stageA.pkl")
if os.path.exists(STAGE_A):
    print("stage A cache found - skipping the fold loop (delete the file to recompute)")
else:
    from joblib import Parallel, delayed
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import roc_auc_score
    def load_rows(path, max_patno=None):
        d = pd.read_csv(path)
        if max_patno is not None:
            d = d[d.PATNO < max_patno]
        d = d.reset_index(drop=True)
        X = pd.DataFrame({f: pd.to_numeric(d[f], errors="coerce") for f in FEATURES})
        X["age"] = d["Age"].astype(float)
        X["sex"] = d["Sex"].map({1: 0, 2: 1}).astype(float)
        y = (d["APPRDX"] == 1).astype(int).values
        return X[COLS], y, d["PATNO"].values

    X, y, g = load_rows(TRAIN_FILE, CFG["new_cohort_min_patno"])
    Xt, yt, gt = load_rows(TEST_FILE, CFG["new_cohort_min_patno"])
    print("train rows", len(X), "patients", len(set(g)), "| test rows", len(Xt), "patients", len(set(gt)), flush=True)

    splits = [(a, b) for r in range(lib.N_REPEATS)
              for a, b in StratifiedGroupKFold(lib.N_SPLITS, shuffle=True, random_state=lib.RANDOM_STATE + r).split(X, y, g)]

    def task(k, tr_i, va_i):
        Xtr = lib.harmonize(X.iloc[tr_i].copy(deep=True), np.array(y[tr_i]), g[tr_i], FEATURES, lib.RANDOM_STATE + k)
        Xva = X.iloc[va_i]
        m1 = {f: lib.fit_m1_feature(Xtr[[f, "age", "sex"]], y[tr_i]) for f in FEATURES}
        return {"k": k, "tr_i": tr_i, "va_i": va_i, "Xtr": Xtr,
                "m1": lib.m1_feature_probs(m1, Xva),
                "m2": lib.predict_m2(lib.fit_m2(Xtr, y[tr_i]), Xva)}

    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS, backend="loky", max_nbytes=None)(delayed(task)(k, a, b) for k, (a, b) in enumerate(splits))
    print(f"folds done {time.time()-t0:.0f}s", flush=True)

    n = len(X)
    s1 = {f: np.zeros(n) for f in FEATURES}; s2 = np.zeros(n); cnt = np.zeros(n)
    for r in res:
        for f in FEATURES:
            s1[f][r["va_i"]] += r["m1"][f]
        s2[r["va_i"]] += r["m2"]; cnt[r["va_i"]] += 1
    feat_p = lib.unit_mean(g, {f: s1[f] / cnt for f in FEATURES})[FEATURES]
    oof = pd.DataFrame(index=feat_p.index)
    oof["M2"] = lib.unit_mean(g, {"p": s2 / cnt})["p"]
    oof["y"] = pd.Series(y).groupby(g).first()
    yp = oof["y"].values
    oof["M1"] = lib.crossfit(lib.fit_xgb_stacker, lib.predict_xgb_stacker, feat_p.values, yp, seed_offset=100)

    # final fit (as train_final) -> M1, M2 on the test set
    Xh = lib.harmonize(X.copy(deep=True), y, g, FEATURES, lib.RANDOM_STATE)
    fitted = [lib.fit_m1_feature(Xh[[f, "age", "sex"]], y) for f in FEATURES]
    m1 = dict(zip(FEATURES, fitted))
    stk = lib.fit_xgb_stacker(feat_p.values, yp)
    m2 = lib.fit_m2(Xh, y)
    fpt = lib.unit_mean(gt, lib.m1_feature_probs(m1, Xt[FEATURES + ["age", "sex"]]))[FEATURES]
    test = pd.DataFrame(index=fpt.index)
    test["M1"] = lib.predict_xgb_stacker(stk, fpt.values)
    test["M2"] = lib.unit_mean(gt, {"p": lib.predict_m2(m2, Xt)})["p"]
    test["y"] = pd.Series(yt).groupby(gt).first()

    pickle.dump(dict(FEATURES=FEATURES, X=X, y=y, g=g, Xt=Xt, yt=yt, gt=gt, folds=res, Xh=Xh,
                     oof=oof, test=test), open(os.path.join(CACHE, "stageA.pkl"), "wb"))
    from sklearn.metrics import roc_auc_score
    print("OOF  AUC  M1 %.4f  M2 %.4f" % (roc_auc_score(yp, oof.M1), roc_auc_score(yp, oof.M2)))
    print("TEST AUC  M1 %.4f  M2 %.4f" % (roc_auc_score(test.y, test.M1), roc_auc_score(test.y, test.M2)), flush=True)
    print("total %.0fs" % (time.time() - t0))

## Part B. Collaborative-filtering library and the 12 variants

One generic `CF` class implements every variant. All statistics (profile means, standardisation, feature weights, RBF width)
are learned from the training rows passed to `fit`, so it is leakage-free inside cross-validation.

* **Profile**: mean over visits of the 30 items + age + sex; missing items filled with the class-balanced training mean, then standardised.
* **Feature weights `wts`**: `none`, `auc` (|AUC-0.5| per item), `auc2` (squared), `top10`, `lr` (|coefficient| of a class-balanced L2 logistic regression, `lrC`).
* **Similarity `sims`**: raw cosine/Pearson (`rawcos`, `rawprs`), weighted Euclidean RBF `euc` = exp(-D/sigma), Gaussian `gau`, Manhattan, standardised cosine/Pearson/Spearman. Several similarities are averaged.
* **Vote `vote`**: `topk` (K most similar, similarity-weighted), `cw` (inverse-class-frequency weights), `bal` (equal numbers per class), `kern` (softmax weights over the K nearest with temperature `tau`, class-weighted).
* **`alpha`**: shrinkage of the vote towards the prior.

The final model is **V4**: `dict(sims=["euc"], K=80, vote="kern", tau=0.05, wts="lr")` (C = 0.1).

In [ ]:
# Generic collaborative-filtering M3 (patient-profile neighbours) with many design options.
# All statistics are learned from the training rows handed to fit(), so it is leakage-free in CV.
# V4 reproduces the pipeline's M3 (min16_lib.fit_m3/predict_m3).
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

def _rowz(A):
    mu = A.mean(1, keepdims=True); sd = A.std(1, keepdims=True); sd[sd == 0] = 1
    return (A - mu) / sd

def _cosm(A, B):
    return (A @ B.T) / np.maximum(np.linalg.norm(A, axis=1, keepdims=True) * np.linalg.norm(B, axis=1, keepdims=True).T, 1e-12)

def _dist(A, B, p):
    if p == 2:
        d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
        return np.sqrt(np.maximum(d2, 0))
    return np.abs(A[:, None, :] - B[None, :, :]).sum(2)

class CF:
    """spec keys:
       sims   : list from {rawcos, rawprs, euc, gau, manh, zcos, zprs, zspr}
       K      : neighbours      vote : topk | bal | cw | kern     alpha: shrink to prior (0 = none)
       wts    : none | auc | top10     demo: multiplier on age/sex columns (1 = as is)
       tau    : kernel temperature (kern)"""
    def __init__(self, **spec):
        self.s = dict(sims=["euc"], K=15, vote="topk", alpha=0.0, wts="none", demo=1.0, tau=0.05, pw=1.0)
        self.s.update(spec)

    # ---- fit ----
    def fit(self, X, y, units):
        s = self.s
        prof = lib.unit_mean(units, X)
        yp = pd.Series(y).groupby(np.asarray(units)).first().loc[prof.index].values.astype(float)
        self.means = lib.balanced_center(prof, yp, how="mean")
        Xi = prof.fillna(self.means).values.astype(float)
        self.cols = list(prof.columns)
        self.mu, self.sd = Xi.mean(0), Xi.std(0); self.sd[self.sd == 0] = 1
        Z = (Xi - self.mu) / self.sd
        w = np.ones(Z.shape[1])
        if s["wts"] in ("auc2", "lr"):
            if s["wts"] == "auc2":
                a = np.array([abs(roc_auc_score(yp, Z[:, j]) - 0.5) * 2 if Z[:, j].std() > 0 else 0 for j in range(Z.shape[1])]) ** 2
            else:
                from sklearn.linear_model import LogisticRegression
                a = np.abs(LogisticRegression(C=s.get("lrC", 0.1), class_weight="balanced", max_iter=2000).fit(Z, yp).coef_[0])
            w = a / max(a.mean(), 1e-9)
        if s["wts"] in ("auc", "top10"):
            a = np.array([abs(roc_auc_score(yp, Z[:, j]) - 0.5) * 2 if Z[:, j].std() > 0 else 0 for j in range(Z.shape[1])])
            if s["wts"] == "auc":
                w = a / max(a.mean(), 1e-9)
            else:
                w = (a >= np.sort(a)[-10]).astype(float)
        for c in ("age", "sex"):
            if c in self.cols:
                w[self.cols.index(c)] *= s["demo"]
        self.w = np.sqrt(w)
        self.raw, self.Z, self.y = Xi, Z * self.w, yp
        self.prior = yp.mean()
        self.nc = np.array([(yp == 0).sum(), (yp == 1).sum()])
        D = _dist(self.Z, self.Z, 1 if "manh" in s["sims"] else 2)
        self.sigma = float(np.median(D[np.triu_indices(len(D), 1)])) + 1e-8
        if "manh" in s["sims"]:
            self.sigma_m = self.sigma
        if "euc" in s["sims"] or "gau" in s["sims"]:
            D2 = _dist(self.Z, self.Z, 2)
            self.sigma = float(np.median(D2[np.triu_indices(len(D2), 1)])) + 1e-8
        return self

    # ---- similarities (each mapped to roughly [0,1]) ----
    def _sim(self, kind, Xq, Zq):
        if kind == "rawcos": return _cosm(Xq, self.raw)
        if kind == "rawprs": return (_rowz(Xq) @ _rowz(self.raw).T) / Xq.shape[1]
        if kind == "euc":    return np.exp(-_dist(Zq, self.Z, 2) / self.sigma)
        if kind == "gau":    return np.exp(-0.5 * (_dist(Zq, self.Z, 2) / self.sigma) ** 2)
        if kind == "manh":
            D = _dist(Zq, self.Z, 1); return np.exp(-D / self.sigma_m)
        if kind == "zcos":   return (_cosm(Zq, self.Z) + 1) / 2
        if kind == "zprs":   return ((_rowz(Zq) @ _rowz(self.Z).T) / Zq.shape[1] + 1) / 2
        if kind == "zspr":
            R = lambda A: _rowz(np.apply_along_axis(rankdata, 1, A))
            return ((R(Zq) @ R(self.Z).T) / Zq.shape[1] + 1) / 2
        raise ValueError(kind)

    def _vote(self, S):
        s = self.s; k = s["K"]; y = self.y; a = s["alpha"]; v = s["vote"]
        S = np.maximum(S, 0) ** s["pw"]
        if v == "topk":
            idx = np.argsort(S, 1)[:, -k:]
            w = np.take_along_axis(S, idx, 1); lab = y[idx]
            ws = w.sum(1); num = (w * lab).sum(1)
            if a > 0: return (num + a * self.prior) / (ws + a)
            return np.where(ws > 0, num / np.where(ws > 0, ws, 1), lab.mean(1))
        if v == "cw":                                   # class-weighted top-K (prior-corrected)
            idx = np.argsort(S, 1)[:, -k:]
            c = (len(y) / (2 * self.nc))[y.astype(int)][idx]
            w = np.take_along_axis(S, idx, 1) * c; lab = y[idx]
            ws = w.sum(1); num = (w * lab).sum(1)
            if a > 0: return (num + a * 0.5 * (ws / k)) / (ws + a * (ws / k))
            return np.where(ws > 0, num / np.where(ws > 0, ws, 1), 0.5)
        if v in ("bal", "kern"):
            out = np.zeros(len(S)); h = max(k // 2, 1)
            if v == "bal":
                sums = []
                for cl in (0, 1):
                    Sc = np.where(self.y[None, :] == cl, S, -1)
                    idx = np.argsort(Sc, 1)[:, -h:]
                    sums.append(np.take_along_axis(S, idx, 1).sum(1))
                s0, s1 = sums
                tot = s0 + s1
                p = np.where(tot > 0, s1 / np.where(tot > 0, tot, 1), 0.5)
                return (p * tot + a * 0.5) / (tot + a) if a > 0 else p
            # kern: Nadaraya-Watson over the K nearest, softmax weights, class-balanced
            idx = np.argsort(S, 1)[:, -k:]
            sv = np.take_along_axis(S, idx, 1); lab = y[idx]
            w = np.exp((sv - sv.max(1, keepdims=True)) / s["tau"])
            c = (len(y) / (2 * self.nc))[lab.astype(int)]
            w = w * c
            p = (w * lab).sum(1) / w.sum(1)
            return (p * 1.0 + a * 0.5) / (1.0 + a) if a > 0 else p
        raise ValueError(v)

    def predict(self, X, units):
        prof = lib.unit_mean(units, X[self.cols])
        Xq = prof.fillna(self.means).values.astype(float)
        Zq = ((Xq - self.mu) / self.sd) * self.w
        p = np.mean([self._vote(self._sim(k, Xq, Zq)) for k in self.s["sims"]], axis=0)
        return pd.Series(p, index=prof.index)

# ---------------- the 12 variants (Table 5; numbered by rank of the ensemble OOF AUC) ----------------
ALL = {
    "V1 LR-weighted Euc, K = 40, class-weighted votes": dict(sims=["euc"], K=40, vote="cw", wts="lr"),
    "V2 LR-weighted Gaussian, K = 60": dict(sims=["gau"], K=60, wts="lr"),
    "V3 LR-weighted Euc, K = 40, softmax tau = 0.1": dict(sims=["euc"], K=40, vote="kern", tau=0.1, wts="lr"),
    "V4 LR-weighted Euc, K = 80, softmax tau = 0.05": dict(sims=["euc"], K=80, vote="kern", tau=0.05, wts="lr"),
    "V5 AUC2-weighted Euc, K = 25": dict(sims=["euc"], K=25, wts="auc2"),
    "V6 LR-weighted Gaussian, K = 25": dict(sims=["gau"], K=25, wts="lr"),
    "V7 LR-weighted (C = 0.03) Euc, K = 25": dict(sims=["euc"], K=25, wts="lr", lrC=0.03),
    "V8 LR-weighted (C = 0.03) Gaussian, K = 40": dict(sims=["gau"], K=40, wts="lr", lrC=0.03),
    "V9 LR-weighted standardised cos + Pearson + Euc, K = 40": dict(sims=["zcos", "zprs", "euc"], K=40, wts="lr"),
    "V10 LR-weighted standardised cos + Pearson + Euc, K = 25": dict(sims=["zcos", "zprs", "euc"], K=25, wts="lr"),
    "V11 LR-weighted (C = 0.3) Euc, K = 25": dict(sims=["euc"], K=25, wts="lr", lrC=0.3),
    "V12 LR-weighted (C = 1) Euc, K = 25": dict(sims=["euc"], K=25, wts="lr", lrC=1.0),
}
FINAL_NAME = [k for k in ALL if k.startswith('V4 ')][0]


In [ ]:
print(len(ALL), "collaborative-filtering variants")
pd.Series({k: str(v) for k, v in list(ALL.items())[:5]}).to_frame("spec")


## Part C. Evaluate every variant inside the ensemble

For each variant and each of the 15 training folds M3 is fitted on the (harmonised) training rows and predicts the held-out rows; the
fold predictions are averaged per participant (out-of-fold, OOF). The cross-fitted meta-learner then combines M1, M2 and the variant M3.
Reported per variant: stand-alone M3 OOF AUC and Brier, **saturation** (share of OOF outputs that are exactly 0 or 1 on the logit-clip scale),
the number of distinct output values, correlation with M1/M2 on the logit scale, ensemble OOF AUC/Brier, and held-out test metrics at the
Youden threshold from OOF.

In [ ]:
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from xgboost import XGBClassifier

A = pickle.load(open(os.path.join(CACHE, "stageA.pkl"), "rb"))
X, y, g, Xt, yt, gt, folds, Xh, oof, test = (A[k] for k in ["X", "y", "g", "Xt", "yt", "gt", "folds", "Xh", "oof", "test"])
yp, ytp = oof["y"].values, test["y"].values

def run_variant(name, spec):
    mk = lambda: CF(**spec)
    parts, tparts = [], []
    for r in folds:
        m = mk().fit(r["Xtr"], y[r["tr_i"]], g[r["tr_i"]])
        parts.append(m.predict(X.iloc[r["va_i"]], g[r["va_i"]]))
        tparts.append(m.predict(Xt, gt))
    o = pd.concat(parts).groupby(level=0).mean().loc[oof.index].values
    tb = pd.concat(tparts).groupby(level=0).mean().loc[test.index].values       # fold-bagged test pred
    full = mk().fit(Xh, y, g).predict(Xt, gt).loc[test.index].values            # pipeline's final-fit test pred
    return name, o, full, tb

def youden(yy, p):
    f, t, th = roc_curve(yy, p); return float(th[np.argmax(t - f)])

def point(yy, p, thr):
    pred = p >= thr; tp = ((pred) & (yy == 1)).sum(); fn = ((~pred) & (yy == 1)).sum()
    tn = ((~pred) & (yy == 0)).sum(); fp = ((pred) & (yy == 0)).sum()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    return se, sp, (se + sp) / 2

def slope(yy, p):
    from sklearn.linear_model import LogisticRegression
    L = lib._lg(p).reshape(-1, 1)
    return float(LogisticRegression(C=1e6, max_iter=1000).fit(L, yy).coef_[0, 0])

def ensemble(cols_oof, cols_test, tag):
    Po, Pt = np.column_stack(cols_oof), np.column_stack(cols_test)
    fo = lib.crossfit(lib.fit_lr_on_logits, lib.predict_lr_on_logits, Po, yp, seed_offset=200)
    thr = youden(yp, fo)
    meta = lib.fit_lr_on_logits(Po, yp)
    ft = lib.predict_lr_on_logits(meta, Pt)
    se, sp, ba = point(ytp, ft, thr)
    so, spo, bao = point(yp, fo, thr)
    return dict(ens_oof_auc=roc_auc_score(yp, fo), ens_oof_ba=bao, ens_oof_brier=brier_score_loss(yp, fo),
                ens_test_auc=roc_auc_score(ytp, ft), ens_test_sens=se, ens_test_spec=sp, ens_test_ba=ba,
                ens_test_brier=brier_score_loss(ytp, ft), ens_test_slope=slope(ytp, ft), thr=thr,
                meta_w=list(np.round(meta.coef_[0], 2))), fo, ft

In [ ]:
t0 = time.time()
jobs = [(n, s) for n, s in ALL.items()]
if QUICK:
    keep_q = ["V1 ", "V2 ", "V4 "]
    jobs = [j for j in jobs if any(j[0].startswith(k) for k in keep_q)]
print(len(jobs), "variants")
res = Parallel(n_jobs=N_JOBS, backend="loky", max_nbytes=None)(delayed(run_variant)(*j) for j in jobs)
print(f"variants fitted {time.time()-t0:.0f}s", flush=True)
base, _, _ = ensemble([oof.M1.values, oof.M2.values], [test.M1.values, test.M2.values], "none")
rows = []
store = {}
for name, o, full, tb in res:
    lo = lib._lg(o)
    d, fo, ft = ensemble([oof.M1.values, oof.M2.values, o], [test.M1.values, test.M2.values, full], name)
    db, _, _ = ensemble([oof.M1.values, oof.M2.values, o], [test.M1.values, test.M2.values, tb], name)
    sat = float(((o <= 2e-4) | (o >= 1 - 2e-4)).mean())
    row = dict(variant=name, m3_oof_auc=roc_auc_score(yp, o), m3_oof_brier=brier_score_loss(yp, o),
               m3_test_auc=roc_auc_score(ytp, full), m3_test_brier=brier_score_loss(ytp, full),
               m3_test_auc_bagged=roc_auc_score(ytp, tb),
               n_unique_oof=len(np.unique(np.round(o, 6))), saturated_frac=sat,
               corr_M1=np.corrcoef(lo, lib._lg(oof.M1.values))[0, 1], corr_M2=np.corrcoef(lo, lib._lg(oof.M2.values))[0, 1],
               ens_test_auc_bagged_m3=db["ens_test_auc"], **d)
    rows.append(row); store[name] = dict(oof=o, full=full, bag=tb, fo=fo, ft=ft)
R = pd.DataFrame(rows)
R["d_ens_oof_auc_vs_noM3"] = R.ens_oof_auc - base["ens_oof_auc"]
R["d_ens_test_auc_vs_noM3"] = R.ens_test_auc - base["ens_test_auc"]
R.to_csv(os.path.join(OUT, "stageB_variants.csv"), index=False)
pickle.dump(dict(R=R, store=store, base=base), open(os.path.join(CACHE, "stageB.pkl"), "wb"))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 38)
print("NO-M3 baseline:", {k: (round(v, 4) if not isinstance(v, list) else v) for k, v in base.items()})
print(R.sort_values("ens_oof_auc", ascending=False)[["variant", "m3_oof_auc", "ens_oof_auc", "ens_oof_ba", "ens_test_auc", "ens_test_ba", "m3_test_auc", "saturated_frac", "corr_M1", "corr_M2"]].round(4).to_string())
print(f"total {time.time()-t0:.0f}s")

## Part D. Paired bootstrap and ranking

* Differences in ensemble OOF AUC are bootstrapped **by participant** (2000 resamples, paired) against the variant selected as final (V4) and against the ensemble without any M3 (M1 + M2 only).
* The 12 variants are ranked by ensemble OOF AUC and saved (`top12_m3_variants.csv`; `table5_label` is the label of Table 5).


In [ ]:
from sklearn.metrics import roc_auc_score
yp, ytp = oof["y"].values, test["y"].values
FIN = FINAL_NAME

R = R.sort_values("ens_oof_auc", ascending=False).reset_index(drop=True)
keep = list(R.variant)

# ---- paired participant bootstrap on OOF ensemble AUC: difference vs the final variant (V4) and vs no M3 (M1 + M2 only)
rng = np.random.RandomState(7); B = 2000; n = len(yp)
idx = [rng.randint(0, n, n) for _ in range(B)]
idx = [i for i in idx if 0 < yp[i].sum() < n]
def auc_b(p): return np.array([roc_auc_score(yp[i], p[i]) for i in idx])
fo_none = lib.crossfit(lib.fit_lr_on_logits, lib.predict_lr_on_logits, np.column_stack([oof.M1, oof.M2]), yp, 200)
a_none, a_fin = auc_b(fo_none), auc_b(store[FIN]["fo"])
rows = []
for v in keep:
    a = auc_b(store[v]["fo"])
    d1, d2 = a - a_fin, a - a_none
    r = R[R.variant == v].iloc[0].to_dict()
    r.update(d_vs_final=float(np.mean(d1)), d_vs_final_lo=float(np.percentile(d1, 2.5)), d_vs_final_hi=float(np.percentile(d1, 97.5)),
             d_vs_noM3=float(np.mean(d2)), d_vs_noM3_lo=float(np.percentile(d2, 2.5)), d_vs_noM3_hi=float(np.percentile(d2, 97.5)))
    rows.append(r)
T = pd.DataFrame(rows).sort_values("ens_oof_auc", ascending=False).reset_index(drop=True)
top12 = T.copy(); top12.insert(0, "rank", range(1, len(top12) + 1))
top12.insert(1, "table5_label", ["V%d" % r for r in top12["rank"]])
top12.to_csv(os.path.join(OUT, "top12_m3_variants.csv"), index=False)
pd.set_option("display.width", 260); pd.set_option("display.max_columns", 50); pd.set_option("display.max_colwidth", 60)
print(top12[["rank", "variant", "m3_oof_auc", "ens_oof_auc", "d_vs_final", "d_vs_final_lo", "d_vs_final_hi", "ens_oof_ba", "ens_test_auc", "ens_test_sens", "ens_test_spec", "ens_test_ba", "ens_test_brier", "m3_test_auc", "saturated_frac"]].round(4).to_string())
print("\nNO-M3 baseline (M1+M2 only):", {k: (round(float(v), 4) if not isinstance(v, list) else v) for k, v in base.items()})
print("single-model OOF AUC: M1 %.4f M2 %.4f" % (roc_auc_score(yp, oof.M1), roc_auc_score(yp, oof.M2)))
print("single-model TEST AUC: M1 %.4f M2 %.4f" % (roc_auc_score(ytp, test.M1), roc_auc_score(ytp, test.M2)))
print("test participants: n=%d PD=%d HC=%d" % (len(ytp), ytp.sum(), (ytp == 0).sum()))


## Part E. Final model (V4) and the Brier-rule check

`min16_lib.fit_m3` / `predict_m3` implement V4; `Train_3SubModels.ipynb` fits the final pipeline with it, and its saved results are read here.
**Rule:** the final ensemble's Brier score must be less than or equal to the smallest Brier score among M1, M2 and M3 (in the same pipeline),
on training OOF predictions and on the held-out test set.

In [ ]:
res = os.path.join(BASE, "results")
rows = []
for an in ["primary", "sensitivity_all_cohorts"]:
    f = os.path.join(res, an, "metrics.csv")
    if not os.path.exists(f): print("missing", f, "- run Train_3SubModels.ipynb first"); continue
    M = pd.read_csv(f)
    for ev in ["train OOF (patient)", "TEST patient level", "TEST single visit"]:
        s = M[(M.Evaluation == ev)]
        b = {m: float(s[s.Model == m].Brier.iloc[0]) for m in ["M1", "M2", "M3", "Final"] if (s.Model == m).any()}
        best = min(b[m] for m in ["M1", "M2", "M3"])
        rows.append(dict(analysis=an, evaluation=ev, **{k + "_Brier": round(v, 4) for k, v in b.items()},
                         best_submodel=round(best, 4), final_le_best=bool(b["Final"] <= best + 1e-9)))
display(pd.DataFrame(rows))